<a href="https://www.kaggle.com/code/harshalsanjivpatil/ai-job-market-insider-and-career-advisor?scriptVersionId=282090423" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

#  **Step 1: Setup & Observability**

**Why we do this: We need to initialize the ADK environment, configure our API keys for the Gemini model, and set up observability. We use the LoggingPlugin from Day 4 to capture "black box" data (prompts, thoughts, tool calls) so we can debug the system.**

In [ ]:
import os
import sys
import time
import subprocess
import requests
import logging

from google.genai import types
from google.adk.agents import Agent, LlmAgent, SequentialAgent, ParallelAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent, AGENT_CARD_WELL_KNOWN_PATH
from kaggle_secrets import UserSecretsClient

# 1. Configure API Key
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("[SUCCESS] API Key Configured")
except Exception as e:
    print(f"[WARNING] Authentication Error: {e}")

# 2. Configure Retry Logic
# Handles temporary API glitches automatically
retry_config = types.HttpRetryOptions(
    attempts=3, exp_base=2, initial_delay=1, http_status_codes=[429, 500, 503]
)

# 3. Configure Observability (Logging)
# This plugin records agent thoughts and actions
logging.basicConfig(level=logging.INFO)
logging_plugin = LoggingPlugin()

print("[SUCCESS] Setup & Observability Complete")

# **Step 2: The Remote "Trend Analyzer" Service (A2A)**



**Why we do this: In Day 5, you learned about the Agent2Agent (A2A) protocol. We use it here to create a specialized "Trend Analyzer" agent that runs as a separate microservice. This decouples complex analysis logic from the main pipeline and simulates integrating with an external vendor or team.** 

**We use to_a2a() to expose the agent as a server and RemoteA2aAgent to connect to it.**


In [ ]:
# Define the code for the separate microservice
trend_analyzer_code = """
import os
from google.adk.agents import LlmAgent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.genai import types

retry_config = types.HttpRetryOptions(attempts=3, exp_base=2, initial_delay=1, http_status_codes=[429, 500, 503])

# The Specialist Agent
trend_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="trend_analyzer",
    description="Analyzes a list of skills to identify market trends.",
    instruction=\"\"\"
    You are a Job Market Trend Analyst.
    Input: A list of technical skills extracted from job descriptions.
    Task: Identify 3 'Rising Stars' (skills gaining popularity) and 3 'Foundational Staples'.
    Output: A concise analysis report.
    \"\"\"
)

# Expose via A2A on port 8001
app = to_a2a(trend_agent, port=8001)
"""

# Write the server code to a file
with open("trend_server.py", "w") as f:
    f.write(trend_analyzer_code)

# Start the server in the background
print("[STARTING] Remote Trend Analyzer Service...")
server_process = subprocess.Popen(
    ["uvicorn", "trend_server:app", "--host", "localhost", "--port", "8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ}
)

# Health check: Wait for the server to come online
for _ in range(15):
    try:
        if requests.get("http://localhost:8001/.well-known/agent-card.json").status_code == 200:
            print("[SUCCESS] Remote Trend Analyzer is Online (A2A)")
            break
    except:
        time.sleep(1)

# Connect to the remote agent using the Client Proxy
remote_trend_analyzer = RemoteA2aAgent(
    name="RemoteTrendAnalyzer",
    agent_card=f"http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}"
)

# **Step 3: The Search Team (Parallel Agents)**


**Why we do this: Searching the web takes time. If we search for "AI Jobs" and then "Data Science Jobs" sequentially, it takes twice as long. From Day 1b, we know ParallelAgent allows multiple agents to run concurrently.**

**We use the Google Search tool so these agents can find real-time market data.**


In [ ]:
# Agent 1: Search for AI Engineering jobs
ai_job_searcher = Agent(
    name="AI_Job_Searcher",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    tools=[google_search],
    instruction="Search for 'most in-demand AI engineering skills 2025'. List the top 5 technical skills found.",
    output_key="ai_skills"  # Saves result to session state
)

# Agent 2: Search for Data Science jobs
data_job_searcher = Agent(
    name="Data_Job_Searcher",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    tools=[google_search],
    instruction="Search for 'top Data Science skills 2025'. List the top 5 technical skills found.",
    output_key="data_skills" # Saves result to session state
)

# The Parallel Team Manager
parallel_search_team = ParallelAgent(
    name="Market_Intelligence_Team",
    sub_agents=[ai_job_searcher, data_job_searcher]
)

print("[SUCCESS] Parallel Search Team Created")

# **Step 4: Processing & Recommendations (Sequential Flow)**

**Why we do this: We need a specific order of operations:**


* **Extract clean skills from the search results**.
* **Analyze those skills using the remote A2A service.**
* **Recommend actions based on the analysis.**

**We use a SequentialAgent to enforce this order. Note how we use AgentTool to wrap the remote A2A agent so our local "Trend_Analysis_Bridge" agent can call it like a function.**


In [ ]:
# Agent 3: Skill Extractor (Cleans the data)
skill_extractor = Agent(
    name="Skill_Extractor",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    Review the findings from the search team:
    AI Skills: {ai_skills}
    Data Skills: {data_skills}
    
    Create a single, de-duplicated comma-separated list of all extracted technical skills.
    """,
    output_key="consolidated_skills"
)

# Agent 4: The Bridge to our Remote Service
# Uses the remote A2A agent as a tool
trend_adapter = Agent(
    name="Trend_Analysis_Bridge",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    tools=[AgentTool(remote_trend_analyzer)], 
    instruction="""
    Take this list of skills: {consolidated_skills}
    
    Call the 'trend_analyzer' tool to analyze these skills. 
    Return the analysis report exactly as provided by the tool.
    """,
    output_key="trend_report"
)

# Agent 5: The Career Advisor
advisor_agent = Agent(
    name="Career_Advisor",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
    Based on this Trend Analysis: 
    {trend_report}
    
    Recommend 3 specific projects a candidate should build to demonstrate these skills.
    Be brief and high-impact.
    """,
    output_key="final_recommendation"
)

# The Master Pipeline
job_market_pipeline = SequentialAgent(
    name="AI_Job_Market_Insider_Pipeline",
    sub_agents=[
        parallel_search_team, # Step 1: Search (Parallel)
        skill_extractor,      # Step 2: Extract
        trend_adapter,        # Step 3: Analyze (Remote A2A)
        advisor_agent         # Step 4: Recommend
    ]
)

print("[SUCCESS] Pipeline Agents Created")

# **Step 5: Execution**

**Why we do this: We use the InMemoryRunner to execute the pipeline. We pass the logging_plugin here so we can see the traces of the parallel execution and the remote A2A calls in the output.**

In [ ]:
print("\n[INITIALIZING] AI Job Market Insider Pipeline...")
runner = InMemoryRunner(
    agent=job_market_pipeline, 
    plugins=[logging_plugin] # Attach Observability
)

# Run the pipeline
# We don't need a complex prompt here because the agents have specific instructions
response = await runner.run_debug("Generate a job market report.")

# Cleanup: Always stop your background processes!
try:
    server_process.terminate()
    print("\n[STOPPED] Remote Server Stopped.")
except:
    pass